In [1]:
# Import libraries
import pandas as pd
import os 
import sys

sys.path.append(os.path.abspath(".."))

import functions.wrangling as wrg

# Set working directory
os.chdir(r"G:\.shortcut-targets-by-id\1qO0AfYMqzVbXreDMm-gZUvrYXtVZCnDA\CHL8010F2  CPCSSN Dataset") 

In [2]:
# Load and clean datasets

# Define and load file paths 
file_paths = {
    'patient': 'C4MPatient.csv',
    'lab': 'C4MLab.csv',
    'diag': 'C4MEncounterdiagnosis.csv',
    'condition': 'C4MHealthCondition.csv'
}
datasets = wrg.load_csv(file_paths)

# Specify columns to keep from each dataset
columns_to_keep = {
    'patient': ["Patient_ID", "Sex", "BirthYear"],
    'lab': ["Patient_ID", "Name_calc", "TestResult_calc", "PerformedDate"],
    'diag': ["Patient_ID", "DiagnosisText_calc", "DiagnosisCode_calc", "DateCreated"],
    'condition': ["Patient_ID", "DiagnosisText_calc", "DateCreated"]
}
datasets = wrg.select_columns(datasets, columns_to_keep)

# Clean diagnosis data 
diagnosis_cleaning_steps = [
    ('DiagnosisText_calc', 'uppercase'),
    ('DiagnosisCode_calc', 'strip'),
    ('DiagnosisCode_calc', 'dropna'),
    ('DateCreated', 'datetime')
]
datasets['diag'] = wrg.replace_string_nan(datasets['diag'], 'DiagnosisCode_calc')
datasets['diag'] = wrg.preprocess_data(datasets['diag'], diagnosis_cleaning_steps)

In [4]:
# Extract bipolar disorder lab results and summarize patient counts

# Define BD ICD-9 codes and relevant markers
bd_codes = ["296.0", "296.1", "296.4", "296.5", "296.6", "296.7", "296.80", "296.89"]
relevant_markers = ["TOTAL CHOLESTEROL", "HBA1C", "HDL", "FASTING GLUCOSE", "LDL", "INR", "GLUCOSE TOLERANCE"]

# Extract lab results that occur after BD diagnosis (filtered by relevant markers)
bd_labs_after = wrg.extract_labs_relative_to_diagnosis(
    lab_df=datasets['lab'],
    diag_df=datasets['diag'],
    diagnosis_codes=bd_codes,
    lab_test_names=relevant_markers
)

# Get the first BD diagnosis date per patient
first_dx = wrg.get_first_diagnosis(datasets['diag'], 'DiagnosisCode_calc', 'DateCreated')
first_dx = first_dx[first_dx['DiagnosisCode_calc'].isin(bd_codes)]
first_dx = first_dx.rename(columns={'DateCreated': 'BD_Diagnosis_Date', 'DiagnosisCode_calc': 'BD_Code'})

# Merge diagnosis info into labs 
bd_labs_after = bd_labs_after.merge(
    first_dx[['Patient_ID', 'BD_Diagnosis_Date', 'BD_Code']],
    on='Patient_ID', how='left'
).drop(columns=['Lab_Timing'])


# Pivot lab data to wide format
bd_labs_after = wrg.pivot_lab_data(
    bd_labs_after,
    index_cols=['Patient_ID', 'PerformedDate', 'BD_Diagnosis_Date', 'BD_Code'],
    name_col='Name_calc',
    value_col='TestResult_calc'
).sort_values(['Patient_ID', 'PerformedDate'])


# Classify patitents by lab timing using only relevant markers
lab_timing_summary_relevant, bd_first_clean = wrg.classify_lab_timing(
    lab_df=datasets['lab'][datasets['lab']['Name_calc'].isin(relevant_markers)],
    diag_df=datasets['diag'],
    diagnosis_codes=bd_codes,
    relevant_markers=None
)

# Count how many BD patients had additional non-BD diagnoses on the same day 
non_bd_same_day_count = wrg.other_dx_same_day(
    diag_df=datasets['diag'],
    bd_first_clean=bd_first_clean,
    bd_codes=bd_codes
)

# Separate lab timing groups
only_before = lab_timing_summary_relevant[lab_timing_summary_relevant['Lab_Data_Timing'] == 'Only before']['Patient_ID']
only_after = lab_timing_summary_relevant[lab_timing_summary_relevant['Lab_Data_Timing'] == 'Only after']['Patient_ID']
both = lab_timing_summary_relevant[lab_timing_summary_relevant['Lab_Data_Timing'] == 'Both']['Patient_ID']

# Count patients with non-BD same-day diagnoses AND lab results after BD diagnosis
nonbd_diag_same_day_after = wrg.non_bd_same_day_with_labs_after(
    diag_df=datasets['diag'],
    bd_labs_after_df=bd_labs_after,
    bd_first_clean=bd_first_clean,
    bd_codes=bd_codes
)

# Final summary
summary_stats = [
    ("All patients have BD as their first-ever diagnosis", bd_first_clean['first_any_dx_code'].isin(bd_codes).all()),
    ("Total BD patients (first-ever diagnosis)", bd_first_clean['Patient_ID'].nunique()),
    ("Patients with ANY lab results", lab_timing_summary_relevant['Patient_ID'].nunique()),
    ("Patients with labs BEFORE diagnosis", pd.concat([only_before, both]).nunique()),
    ("Patients with labs AFTER diagnosis", pd.concat([only_after, both]).nunique()),
    ("Patients with labs BOTH before and after diagnosis", both.nunique()),
    ("Number of patients with other diagnoses on the same day", non_bd_same_day_count),
    ("Patients with lab AFTER BD diagnosis and non-BD diagnosis on same day", nonbd_diag_same_day_after),
    ("Shape of the lab dataset after filtering", bd_labs_after.shape)
]
wrg.print_summary_stats(summary_stats)

All patients have BD as their first-ever diagnosis: True
Total BD patients (first-ever diagnosis): 378
Patients with ANY lab results: 218
Patients with labs BEFORE diagnosis: 22
Patients with labs AFTER diagnosis: 214
Patients with labs BOTH before and after diagnosis: 18
Number of patients with other diagnoses on the same day: 105
Patients with lab AFTER BD diagnosis and non-BD diagnosis on same day: 71
Shape of the lab dataset after filtering: (1196, 11)
